# Ontario Winters & El Niño — EDA notebook

This notebook runs the same pipeline as `python/run_pipeline.py` and shows every EDA chart inline,
grouped by script. Each chart is also saved as a PNG in the run's `figures/` folder, so the notebook
and the pipeline always produce identical charts.

**How to use:** open from the `notebooks/` folder and choose *Run All*. The first run takes about a minute.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Find the repo root (the folder that contains python/config.py).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'python' / 'config.py').is_file())
PY = ROOT / 'python'

# Let Python find config.py and the scripts inside the subfolders.
sys.path.insert(0, str(PY))
for sub in ['01_collection', '02_cleaning', '03_analysis', '04_eda']:
    sys.path.insert(0, str(PY / sub))

import run_pipeline
from eda import create_charts
from extra_charts import latitude_gradient, index_vs_anomaly, usability_grid, oni_vs_roni
from eda_scratch import create_scratch_charts
from eda_impacts import (heating_demand_by_city, freeze_thaw_vs_maple, enso_vs_freeze_thaw,
                         growing_season_by_city, gdp_vs_enso)
from viz_style import apply_style
apply_style()

ImportError: cannot import name 'latitude_gradient' from 'extra_charts' (k:\Npower\IBM Data Analyst\Projects\Ontario El Nino\ontario-elnino-winters\python\04_eda\extra_charts.py)

## 1. Run the pipeline (data only)

Cleans the stations, calculates the KPIs and runs the bootstrap tests. `--no-plots` skips the charts here,
because the cells below draw them one section at a time. Results are written to `runs/<timestamp>/`.

In [ ]:
results = run_pipeline.main(['--no-plots'])
figures = results['output'] / 'figures'
figures.mkdir(exist_ok=True)
winters, comparison, trends = results['winters'], results['comparison'], results['trends']

## 2. Core charts (`eda.py`) — business questions BQ1–BQ6

For each KPI (mean temperature, temperature anomaly, total snowfall, snow days, very cold days):
a box plot by ENSO class per city, then the El Niño − Neutral differences with 95% intervals.
Ends with the anomaly timeline and the data-coverage chart.

In [ ]:
create_charts(results['daily'], winters, comparison, trends, figures)

## 3. Extra charts (`extra_charts.py`)

### North–south (latitude) gradient — BQ4

In [ ]:
latitude_gradient(comparison, figures)

### ENSO index strength vs. temperature anomaly

In [ ]:
index_vs_anomaly(winters, figures)

### Which winters are usable (data-quality grid)

In [ ]:
usability_grid(winters, figures)

### ONI vs. RONI — does the choice of index change the results?

In [ ]:
oni_vs_roni(comparison, figures)

## 4. Exploratory charts (`eda_scratch.py`)

Quick views for exploring. Error bars here are ±1 standard error, not bootstrap intervals —
use `enso_comparisons.csv` for significance.

In [ ]:
create_scratch_charts(winters, comparison, trends, figures)

## 5. Impact charts (`eda_impacts.py`)

### Heating demand (heating-degree-days) by city

In [ ]:
heating_demand_by_city(winters, figures)

### Sap-season freeze-thaw days vs. maple production

**Caveat:** the second cell tests whether El Niño changes freeze-thaw days. If its interval includes zero,
the maple link cannot be attributed to El Niño.

In [ ]:
if results['maple'] is not None:
    freeze_thaw_vs_maple(results['sap_season'], results['maple'], figures)
else:
    print('Maple syrup file not found - skipped')

In [ ]:
enso_vs_freeze_thaw(results['sap_season'])

### Growing-season length by city

In [ ]:
growing_season_by_city(results['growing_season'], figures)

### Ontario GDP growth vs. ENSO (scope boundary)

In [ ]:
if results['gdp'] is not None:
    gdp_vs_enso(results['gdp'], figures)
else:
    print('GDP file not found - skipped')

## Where everything is saved

In [ ]:
print(f"Run folder: {results['output']}")
print(f"{len(list(figures.glob('*.png')))} charts saved in {figures}")